## Import

In [ ]:
import pandas as pd
import random
import os
import numpy as np

from matplotlib import pyplot as plt

In [ ]:
class CFG:
    SEED = 42

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
seed_everything(CFG.SEED) # Seed 고정

## Data Load

In [ ]:
X_train = pd.read_csv('data/X_train_new.csv')
X_test = pd.read_csv('data/X_test_new.csv')
train = pd.read_csv('data/train.csv')
submit = pd.read_csv('data/sample_submission.csv')
y_train = train['class']

### Model_Baseline

In [ ]:
from sklearn.feature_selection import SelectPercentile
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, ShuffleSplit, StratifiedKFold

In [ ]:
from sklearn.utils import all_estimators
estimators = all_estimators(type_filter='classifier')
names = []
names_list = []
for name, ClassifierClass in estimators:
    names.append(name)
    names_list.append(name)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import CategoricalNB
from sklearn.multioutput import ClassifierChain
from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.tree import ExtraTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.gaussian_process.gpc import GaussianProcessClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.semi_supervised import LabelPropagation
from sklearn.semi_supervised import LabelSpreading
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.neural_network.multilayer_perceptron import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import NearestCentroid
from sklearn.svm import NuSVC
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multiclass import OutputCodeClassifier
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.linear_model import Perceptron
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neighbors import RadiusNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import VotingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
for i in range(len(names)):
  names[i] = eval(names[i])

In [ ]:
print(names[:5])
print(names_list[:5])

In [ ]:
"""
Random_state 고정(O):
정상 작동 : 0,1,7,8,9,10,12,13,14,19,22,26,30,31,34,35,37,38
정상 작동,파라미터추가(solver='liblinear') : 20,21

Random_state 고정(X)
random_state 고정X: 2,3,11,15,16,17,18,25,32,36

다른모델 사용(보완모델):
base_estimator(random_state) 오류 : 5
estimator 오류(random_state) : 29
estimator 오류(random_state_X) : 23, 27(다수결 투표),28, 39(stacking), 40(voting)

작동오류:
오류(작동X) : 4(value값에 음수),6(value값에 음수),24(value값에 음수)
dataset오류 : 33(이웃간 거리계산X)
"""

In [ ]:
random_state_on = [0,1,7,8,9,10,12,13,14,19,22,26,30,31,34,35,37,38]
random_state_on_lib = [20,21]
random_state_off = [2,3,11,15,16,17,18,25,32,36]

In [ ]:
model_list = []
feature_per = []
cv_scr = []

for i in range(len(random_state_on)):
  # 사용할 모델 설정
  model = names[random_state_on[i]](random_state=CFG.SEED)

  # 각 특성과 타깃(class) 사이에 유의한 통계적 관계가 있는지 계산하여 특성을 선택하는 방법 
  cv_scores = []
  for p in tqdm(range(10,100,1)):
      X_new = SelectPercentile(percentile=p).fit_transform(X_train, y_train)
      skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=CFG.SEED)    
      cv_score = cross_val_score(model, X_new, y_train, scoring='f1_macro', cv=skf).mean()
      cv_scores.append((p,cv_score))

  # Fiting the best percentile
  best_score = cv_scores[np.argmax([score for _, score in cv_scores])]
  print(names_list[random_state_on[i]], best_score)
  feature_per.append(cv_scores[np.argmax([score for _, score in cv_scores])][0])
  cv_scr.append(cv_scores[np.argmax([score for _, score in cv_scores])][1].tolist())
  model_list.append(names_list[random_state_on[i]])
  model.fit(X_train, y_train)
  submit[f'{names_list[random_state_on[i]]}'] = model.predict(X_test)

In [ ]:
All_Models_01 = pd.DataFrame(zip(feature_per, cv_scr), index = pd.Index(model_list), columns = ['feature_per', 'cv_scr'])

In [ ]:
model_list = []
feature_per = []
cv_scr = []

for i in range(len(random_state_on_lib)):
  # 사용할 모델 설정
  model = names[random_state_on_lib[i]](solver='liblinear', random_state=CFG.SEED)

  # 각 특성과 타깃(class) 사이에 유의한 통계적 관계가 있는지 계산하여 특성을 선택하는 방법 
  cv_scores = []
  for p in tqdm(range(10,100,1)):
      X_new = SelectPercentile(percentile=p).fit_transform(X_train, y_train)
      skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=CFG.SEED)    
      cv_score = cross_val_score(model, X_new, y_train, scoring='f1_macro', cv=skf).mean()
      cv_scores.append((p,cv_score))

  # Fiting the best percentile
  best_score = cv_scores[np.argmax([score for _, score in cv_scores])]
  print(names_list[random_state_on_lib[i]], best_score)
  feature_per.append(cv_scores[np.argmax([score for _, score in cv_scores])][0])
  cv_scr.append(cv_scores[np.argmax([score for _, score in cv_scores])][1].tolist())
  model_list.append(names_list[random_state_on_lib[i]])
  model.fit(X_train, y_train)
  submit[f'{names_list[random_state_on_lib[i]]}'] = model.predict(X_test)

In [ ]:
All_Models_02 = pd.DataFrame(zip(feature_per, cv_scr), index = pd.Index(model_list), columns = ['feature_per', 'cv_scr'])

In [ ]:
model_list = []
feature_per = []
cv_scr = []

for i in range(len(random_state_off)):
  # 사용할 모델 설정
  model = names[random_state_off[i]]()

  # 각 특성과 타깃(class) 사이에 유의한 통계적 관계가 있는지 계산하여 특성을 선택하는 방법 
  cv_scores = []
  for p in tqdm(range(10,100,1)):
      X_new = SelectPercentile(percentile=p).fit_transform(X_train, y_train)
      skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=CFG.SEED)    
      cv_score = cross_val_score(model, X_new, y_train, scoring='f1_macro', cv=skf).mean()
      cv_scores.append((p,cv_score))

  # Fiting the best percentile
  best_score = cv_scores[np.argmax([score for _, score in cv_scores])]
  print(names_list[random_state_off[i]], best_score)
  feature_per.append(cv_scores[np.argmax([score for _, score in cv_scores])][0])
  cv_scr.append(cv_scores[np.argmax([score for _, score in cv_scores])][1].tolist())
  model_list.append(names_list[random_state_off[i]])
  model.fit(X_train, y_train)
  submit[f'{names_list[random_state_off[i]]}'] = model.predict(X_test)

In [ ]:
All_Models_03 = pd.DataFrame(zip(feature_per, cv_scr), index = pd.Index(model_list), columns = ['feature_per', 'cv_scr'])

In [ ]:
names_bst = ['LGBMClassifier','XGBClassifier','CatBoostClassifier']
names_bst_list = ['LGBMClassifier','XGBClassifier','CatBoostClassifier']
for i in range(len(names_bst)):
  names_bst[i] = eval(names_bst[i])
boosting = [0,1,2]

In [ ]:
model_list = []
feature_per = []
cv_scr = []

for i in range(len(boosting)):
  # 사용할 모델 설정
  model = names_bst[boosting[i]](random_state=CFG.SEED)

  # 각 특성과 타깃(class) 사이에 유의한 통계적 관계가 있는지 계산하여 특성을 선택하는 방법 
  cv_scores = []
  for p in tqdm(range(10,100,1)):
      X_new = SelectPercentile(percentile=p).fit_transform(X_train, y_train)
      skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=CFG.SEED)    
      cv_score = cross_val_score(model, X_new, y_train, scoring='f1_macro', cv=skf).mean()
      cv_scores.append((p,cv_score))

  # Fiting the best percentile
  best_score = cv_scores[np.argmax([score for _, score in cv_scores])]
  print(names_bst_list[boosting[i]], best_score)
  feature_per.append(cv_scores[np.argmax([score for _, score in cv_scores])][0])
  cv_scr.append(cv_scores[np.argmax([score for _, score in cv_scores])][1].tolist())
  model_list.append(names_bst_list[boosting[i]])
  model.fit(X_train, y_train)
  submit[f'{names_bst_list[boosting[i]]}'] = model.predict(X_test)

In [ ]:
All_Models_04 = pd.DataFrame(zip(feature_per, cv_scr), index = pd.Index(model_list), columns = ['feature_per', 'cv_scr'])

In [ ]:
all_models_score = pd.concat([All_Models_01,All_Models_02,All_Models_03,All_Models_04]).sort_values(by = 'cv_scr', ascending=False)

In [ ]:
submit = submit.drop(labels='class',axis=1)

In [ ]:
# all_models_score.to_csv('data/models_score.csv')
# submit.to_csv('data/all_submit.csv', index=False)

## Emsemble

In [ ]:
all_models_score = all_models_score.reset_index()

In [ ]:
all_models_score

In [ ]:
submit_01 = submit.drop(labels=['id','DummyClassifier','LinearDiscriminantAnalysis'],axis=1)

In [ ]:
Last_submmit = pd.DataFrame(submit['id'])
Last_submmit['class'] = submit_01.agg(pd.Series.mode,axis=1)

In [ ]:
# submit_01['sum'] = submit_01.sum(axis=1)
# submit_01['A'] = submit_01['sum'].apply(lambda x: x.count('A') )
# submit_01['B'] = submit_01['sum'].apply(lambda x: x.count('B') )
# submit_01['C'] = submit_01['sum'].apply(lambda x: x.count('C') )

In [ ]:
# submit_abc = submit_01.iloc[:,-3:]

In [ ]:
# Last_submmit.to_csv('submissions/Fine_01.csv', index=False)